# Do Our Markets Look Real? Stylized Facts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/convexpi/missions/blob/main/lectures/stylized_facts.ipynb)

Real asset returns aren't Gaussian white noise. They share a handful of robust **stylized facts**
(Cont, 2001): heavy tails, near-zero autocorrelation of returns but **persistent** autocorrelation of
*absolute/squared* returns (volatility clustering), and a gain/loss asymmetry. A simulator — or a
synthetic market you grade on — is only as honest as the stylized facts it reproduces. We use
[`finmlsim`](https://github.com/convexpi/finmlsim) to generate returns with and without these facts
and measure the difference.

## Setup

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q "finmlsim[analysis]"
import numpy as np
import matplotlib.pyplot as plt
import finmlsim as fms
print('ready — finmlsim', fms.__version__)

## 1. Fat tails

A Gaussian says a 5-sigma day is a once-in-a-millennium event; markets have them every few years. A
GARCH process with Student-t innovations produces the heavy tails we actually see.

In [ ]:
gauss = fms.simulate.gaussian(n=5000, seed=0)          # thin-tailed control
garch = fms.simulate.garch(n=5000, dist="t", seed=0)   # clustering + fat tails

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for r, name, c in [(gauss, "Gaussian", "steelblue"), (garch, "GARCH-t", "crimson")]:
    z = (r - r.mean()) / r.std()
    ax[0].hist(z, bins=120, density=True, histtype="step", lw=1.5, label=name, color=c)
ax[0].set_yscale("log"); ax[0].set_title("Return distribution (log scale)"); ax[0].legend()
ax[0].set_xlabel("standardised return")
from scipy import stats
stats.probplot(garch, dist="norm", plot=ax[1]); ax[1].set_title("GARCH-t vs Normal Q-Q")
plt.tight_layout(); plt.show()
print(f"excess kurtosis — Gaussian {fms.stylized.summary(gauss)['excess_kurtosis']:.2f}, "
      f"GARCH-t {fms.stylized.summary(garch)['excess_kurtosis']:.2f}  (0 = normal)")

## 2. Volatility clustering

"Large changes tend to be followed by large changes." Returns themselves are almost uncorrelated
(you can't predict tomorrow's *direction*), but their **magnitude** is highly autocorrelated — so
you *can* predict tomorrow's *volatility*.

In [ ]:
def acf(x, lags=40):
    x = x - x.mean(); denom = np.sum(x * x)
    return np.array([np.sum(x[k:] * x[:-k]) / denom for k in range(1, lags + 1)])

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(np.arange(1, 41) - 0.15, acf(garch), width=0.3, label="returns", color="steelblue")
ax.bar(np.arange(1, 41) + 0.15, acf(np.abs(garch)), width=0.3, label="|returns|", color="crimson")
ax.axhline(0, color="k", lw=0.6); ax.set_xlabel("lag (days)"); ax.set_ylabel("autocorrelation")
ax.set_title("GARCH-t: returns ≈ uncorrelated, |returns| persistent (clustering)"); ax.legend()
plt.tight_layout(); plt.show()

## 3. Scoring the facts

`finmlsim.stylized.summary` reduces a series to the numbers that matter. A realistic market has
**excess kurtosis > 0**, **return autocorrelation ≈ 0**, and **squared-return autocorrelation > 0**.

In [ ]:
import pandas as pd
tbl = pd.DataFrame({"Gaussian": fms.stylized.summary(gauss), "GARCH-t": fms.stylized.summary(garch)})
print(tbl.to_string(float_format=lambda v: f"{v:.4f}"))

## Why this matters for ConvexPi

The Lab's synthetic market already has fat tails and regime vol; its **realistic mode**
(`SyntheticMarket(idio_process="garch")`, powered by this same `finmlsim` GARCH) adds genuine
volatility *clustering* on top — so a strategy meets the same risk texture as real markets before it
ever touches live data.

## Takeaways

1. **Gaussian is a lie for returns** — real tails are heavy; size your risk for 5-sigma days.
2. **Direction is ~unpredictable, volatility is not** — clustering is the most exploitable stylized fact.
3. **Measure, don't assume** — `stylized.summary` tells you whether a simulated (or real) series is honest.

→ These facts underpin risk in every mission; the clustering point returns in Mission 8 (cost of trading).